In [ ]:
import pandas as pd
import re
import regex  # 이모지 제거용

# ============================================
# 1) CSV 불러오기 (파일 경로만 바꿔서 사용)
# ============================================
FILE_PATH = "./reddit_추가/2차_현정3_No번역셀렉.csv"   
OUT_PATH  = "./reddit_추가/2차_현정3_No번역셀렉_cleand.csv"

# 인코딩 문제 방지용
try:
    df = pd.read_csv(FILE_PATH, encoding="utf-8-sig")
except UnicodeDecodeError:
    df = pd.read_csv(FILE_PATH, encoding="cp949")

print("원본 행 개수:", len(df))
print("컬럼 목록:", df.columns.tolist())
display(df.head())

In [ ]:
df.info()

In [ ]:
pip install googletrans==4.0.0-rc1 tqdm

In [ ]:
import pandas as pd
from deep_translator import GoogleTranslator
from tqdm import tqdm
import time

# ===========================================================
# 1. 파일 불러오기
# ===========================================================
FILE_PATH = "./reddit_추가/2차_현정3_No번역셀렉.csv"   
OUT_PATH  = "./reddit_추가/2차_현정3_번역완료_cleand.csv"
try:
    df = pd.read_csv(FILE_PATH, encoding="utf-8-sig")
except UnicodeDecodeError:
    df = pd.read_csv(FILE_PATH, encoding="cp949")

print(f"📂 파일 로드 완료: {len(df)}개 행")

# ===========================================================
# 2. 번역 함수 정의 (긴 글 분할 처리 추가)
# ===========================================================
def translate_content(text):
    # 빈 값 체크
    if pd.isna(text) or str(text).strip() == "":
        return ""
    
    text_str = str(text)
    
    # 💡 핵심 수정: 4500자 이상이면 쪼개서 번역 (안전하게 4500자로 설정)
    if len(text_str) > 4500:
        chunks = [text_str[i:i+4500] for i in range(0, len(text_str), 4500)]
        translated_chunks = []
        
        for chunk in chunks:
            try:
                # 잘린 조각 번역
                res = GoogleTranslator(source='auto', target='ko').translate(chunk)
                translated_chunks.append(res)
                time.sleep(0.5) # 긴 글 처리 중이니 조금 더 쉼
            except Exception as e:
                print(f"⚠️ 긴 글 부분 번역 실패: {e}")
                translated_chunks.append(chunk) # 실패하면 원본 그대로 붙임
        
        return " ".join(translated_chunks) # 조각들을 다시 하나로 합침

    else:
        # 4500자 이하면 한 번에 번역
        try:
            return GoogleTranslator(source='auto', target='ko').translate(text_str)
        except Exception as e:
            print(f"⚠️ 번역 실패: {e}")
            return text_str

# ===========================================================
# 3. 번역 실행
# ===========================================================
print("🚀 번역 시작 (긴 글 자동 분할 기능 적용)...")
tqdm.pandas()

results = []
# tqdm으로 진행 상황 보기
for text in tqdm(df['content']):
    res = translate_content(text)
    results.append(res)
    time.sleep(0.2) # 기본 딜레이

# 번역 결과를 컬럼에 저장
df['content_한글'] = results

# ===========================================================
# 4. 컬럼 정리 및 저장
# ===========================================================
print("\n🧹 컬럼 정리 중...")

target_cols = ['keyword', 'title_한글', 'content_한글', 'date', 'url']
available_cols = [c for c in target_cols if c in df.columns]
final_df = df[available_cols].copy()

# 이름 변경 (한글 꼬리표 떼기)
final_df = final_df.rename(columns={
    'title_한글': 'title',
    'content_한글': 'content'
})

print("\n[최종 결과 미리보기]")
print(final_df.head())

final_df.to_csv(OUT_PATH, index=False, encoding="utf-8-sig")
print(f"\n✅ 저장 완료: {OUT_PATH}")

In [ ]:
# pip install --upgrade httpx httpcore

In [ ]:
# pip install deep-translator tqdm

In [ ]:
final_df